In [24]:
import copy
import os
import pandas as pd
from utilities.data_manager import DataManager
from utilities.backtest_analysis import BacktestAnalysis
from strategies import envelope

In [25]:
bitget = DataManager(name="bitget")

BACKTEST_IGNORED_ATTRIBUTES = [
    "__dict__",
    "__doc__",
    "__module__",
    "__weakref__",
    "bad_trades",
    "best_trade",
    "data",
    "good_trades",
    "trades",
    "wallet",
    "worst_trade",
]


def envelope_backtest(setup):
    ticker = setup["ticker"]
    timeframe = setup["timeframe"]
    start_date = setup["start_date"]
    end_date = setup["end_date"] if "end_date" in setup else None
    strategy_params = setup["strategy_params"]
    leverage = setup["leverage"]

    # On load avec symbole, timeframe, dates,
    ohlcv = bitget.load(ticker, timeframe, start_date, end_date)
    envelopeStrategy = envelope.Strategy(strategy_params, ohlcv)
    envelopeStrategy.run_backtest(
        initial_balance=1000,
        leverage=leverage,
        open_fee_rate=0.0002,
        close_fee_rate=0.0006,
    )
    backtest = BacktestAnalysis(envelopeStrategy)
    return toDataFrame(setup, backtest)



def toDataFrame(setup, backtest):
    backtestData = {
        "avg_fee": backtest.avg_fee,
        "avg_pnl_pct_bad_trades": backtest.avg_pnl_pct_bad_trades,
        "avg_pnl_pct_good_trades": backtest.avg_pnl_pct_good_trades,
        "avg_pnl_pct": backtest.avg_pnl_pct,
        "biggest_fee": backtest.biggest_fee,
        "calmar_ratio": backtest.calmar_ratio,
        "final_balance": backtest.final_balance,
        "global_win_rate": backtest.global_win_rate,
        "hodl_pct": backtest.hodl_pct,
        "initial_balance": backtest.initial_balance,
        "max_drawdown_equity": backtest.max_drawdown_equity,
        "max_drawdown_trades": backtest.max_drawdown_trades,
        "max_lose_streak": backtest.max_lose_streak,
        "max_win_streak": backtest.max_win_streak,
        "mean_bad_trades_duration": backtest.mean_bad_trades_duration,
        "mean_good_trades_duration": backtest.mean_good_trades_duration,
        "mean_trade_duration": backtest.mean_trade_duration,
        "mean_trades_per_day": backtest.mean_trades_per_day,
        "performance_vs_hodl": backtest.performance_vs_hodl,
        "profit_factor": backtest.profit_factor,
        "return_over_max_drawdown": backtest.return_over_max_drawdown,
        "roi": backtest.roi,
        "sharpe_ratio": backtest.sharpe_ratio,
        "sortino_ratio": backtest.sortino_ratio,
        "time_in_position_ratio": backtest.time_in_position_ratio,
        "total_bad_trades": backtest.total_bad_trades,
        "total_fee": backtest.total_fee,
        "total_good_trades": backtest.total_good_trades,
        "total_trades": backtest.total_trades,
    }
    data = copy.deepcopy(setup)
    # flatten strategy params
    del data["strategy_params"]
    data.update(setup["strategy_params"])
    data.update(backtestData)
    data.update(
        {
            "start_date": backtest.wallet.index[0],
            "end_date": backtest.wallet.index[-1],
        }
    )
    data.update({(f"envelope {i+1}", None) for i in range(5)})
    data.update(
        {
            (f"envelope {i+1}", v)
            for i, v in enumerate(setup["strategy_params"]["envelopes"])
        }
    )
    return data


def BacktestsToCSVFile(backtests, path):
    df = pd.DataFrame(backtests)
    # print(df.columns)
    columns = [
        "ticker",
        "timeframe",
        "average_period",
        "average_type",
        "stop_loss_pct",
        "price_jump_pct",
        "leverage",
        "mode",
        "envelope 1",
        "envelope 2",
        "envelope 3",
        "envelope 4",
        "envelope 5",
        "start_date",
        "end_date",
        "initial_balance",
        "final_balance",
        "roi",
        "hodl_pct",
        "global_win_rate",
        "sharpe_ratio",
        "max_drawdown_equity",
        "max_drawdown_trades",
        "total_trades",
        "total_good_trades",
        "total_bad_trades",
        "time_in_position_ratio",
        "avg_pnl_pct",
        "avg_pnl_pct_good_trades",
        "avg_pnl_pct_bad_trades",
        "avg_fee",
        "biggest_fee",
        "calmar_ratio",
        "total_fee",
        "max_lose_streak",
        "max_win_streak",
        "mean_trade_duration",
        "mean_good_trades_duration",
        "mean_bad_trades_duration",
        "mean_trades_per_day",
        # "envelopes",
        # "performance_vs_hodl",
        # "position_size_percentage",
        # "profit_factor",
        # "return_over_max_drawdown",
        # "sortino_ratio",
    ]
    mode = 'a' if os.path.exists(path) else 'w'
    header = not os.path.exists(path)
    df.to_csv(path, columns=columns, index=False, mode=mode, header=header)

In [26]:
# gen avg type and period
AVG_TYPES = [ 'SMA', 'EMA', 'WMA', 'DCM' ]
AVG_PERIODS = range(4, 11)
avgs = []
for avgType in AVG_TYPES:
    for avgPeriod in AVG_PERIODS:
        avgs.append((avgType, avgPeriod))
print(avgs)


[('SMA', 4), ('SMA', 5), ('SMA', 6), ('SMA', 7), ('SMA', 8), ('SMA', 9), ('SMA', 10), ('EMA', 4), ('EMA', 5), ('EMA', 6), ('EMA', 7), ('EMA', 8), ('EMA', 9), ('EMA', 10), ('WMA', 4), ('WMA', 5), ('WMA', 6), ('WMA', 7), ('WMA', 8), ('WMA', 9), ('WMA', 10), ('DCM', 4), ('DCM', 5), ('DCM', 6), ('DCM', 7), ('DCM', 8), ('DCM', 9), ('DCM', 10)]


In [27]:
# Setup de base pour les coins, surtout pour la starting date
DOGE = {
    "ticker": "DOGE/USDT:USDT",
    "timeframe": "5m",
    "start_date": "2023-01-01 00:00:00",
    # "end_date": "2024-01-01 00:00:00",
    "leverage": 1,
    "strategy_params": {
        "average_type": "DCM",  # 'SMA', 'EMA', 'WMA', 'DCM'
        "average_period": 5,
        "envelopes": [0.03, 0.05, 0.07, 0.09],
        "stop_loss_pct": 0.45,
        "price_jump_pct": 0.2,
        "position_size_percentage": 100,  #  % of the balance spread equally across each envelope
        # 'position_size_fixed_amount': 1000,  # fixed amount spread equally across each envelope
        # 'mode': "long"#, "short", 'both' (default)
        # "mode": "long",
    },
}
TAO = {
    "ticker": "TAO/USDT:USDT",
    "timeframe": "5m",
    "start_date": "2024-04-12 00:00:00",
    # "end_date": "2024-01-01 00:00:00",
    "leverage": 1,
    "strategy_params": {
        "average_type": "DCM",  # 'SMA', 'EMA', 'WMA', 'DCM'
        "average_period": 5,
        "envelopes": [0.03, 0.05, 0.07, 0.09],
        "stop_loss_pct": 0.45,
        "price_jump_pct": 0.2,
        "position_size_percentage": 100,  #  % of the balance spread equally across each envelope
        # 'position_size_fixed_amount': 1000,  # fixed amount spread equally across each envelope
        # 'mode': "long"#, "short", 'both' (default)
        # "mode": "long",
    },
}
XRP = {
    "ticker": "XRP/USDT:USDT",
    "timeframe": "5m",
    "start_date": "2020-01-01 05:00:00",
    # "end_date": "2024-01-01 00:00:00",
    "leverage": 1,
    "strategy_params": {
        "average_type": "DCM",  # 'SMA', 'EMA', 'WMA', 'DCM'
        "average_period": 5,
        "envelopes": [0.03, 0.05, 0.07, 0.09],
        "stop_loss_pct": 0.45,
        "price_jump_pct": 0.2,
        "position_size_percentage": 100,  #  % of the balance spread equally across each envelope
        # 'position_size_fixed_amount': 1000,  # fixed amount spread equally across each envelope
        # 'mode': "long"#, "short", 'both' (default)
        # "mode": "long",
    },
}
FET = {
    # pas de DCM, ni WMA
    # periode assez longue > 6+
    "ticker": "FET/USDT:USDT",
    "timeframe": "5m",
    "start_date": "2023-03-03 00:00:00",
    # "end_date": "2024-01-01 00:00:00",
    "leverage": 1,
    "strategy_params": {
        "average_type": "DCM",  # 'SMA', 'EMA', 'WMA', 'DCM'
        "average_period": 5,
        "envelopes": [0.03, 0.05, 0.07, 0.09],
        "stop_loss_pct": 0.45,
        "price_jump_pct": 0.2,
        "position_size_percentage": 100,  #  % of the balance spread equally across each envelope
        # 'position_size_fixed_amount': 1000,  # fixed amount spread equally across each envelope
        # 'mode': "long"#, "short", 'both' (default)
        # "mode": "long",
    },
}
INJ = {
    "ticker": "INJ/USDT:USDT",
    "timeframe": "5m",
    "start_date": "2023-03-17 15:00:00",
    # "end_date": "2024-01-01 00:00:00",
    "leverage": 1,
    "strategy_params": {
        "average_type": "DCM",  # 'SMA', 'EMA', 'WMA', 'DCM'
        "average_period": 5,
        "envelopes": [0.03, 0.05, 0.07, 0.09],
        "stop_loss_pct": 0.45,
        "price_jump_pct": 0.2,
        "position_size_percentage": 100,  #  % of the balance spread equally across each envelope
        # 'position_size_fixed_amount': 1000,  # fixed amount spread equally across each envelope
        # 'mode': "long"#, "short", 'both' (default)
        # "mode": "long",
    },
}
ONDO = {
    "ticker": "ONDO/USDT:USDT",
    "timeframe": "5m",
    "start_date": "2024-03-27 11:10:00",
    # "end_date": "2024-01-01 00:00:00",
    "leverage": 1,
    "strategy_params": {
        "average_type": "DCM",  # 'SMA', 'EMA', 'WMA', 'DCM'
        "average_period": 5,
        "envelopes": [0.03, 0.05, 0.07, 0.09],
        "stop_loss_pct": 0.45,
        "price_jump_pct": 0.2,
        "position_size_percentage": 100,  #  % of the balance spread equally across each envelope
        # 'position_size_fixed_amount': 1000,  # fixed amount spread equally across each envelope
        # 'mode': "long"#, "short", 'both' (default)
        # "mode": "long",
    },
}


In [28]:
AVERAGES = [
    #    ("SMA", 4),
    # ("SMA", 5),
    # ("SMA", 6),
    # ("SMA", 7),
    # ("SMA", 8),
    # ("SMA", 9),
    # ("SMA", 10),
    # ("SMA", 11),
    #    ("EMA", 4),
    # ("EMA", 5),
    # ("EMA", 6),
    # ("EMA", 7),
    # ("EMA", 8),
    # ("EMA", 9),
    ("EMA", 10),
    # ("EMA", 11),
    # ("WMA", 4),
    # ("WMA", 5),
    # ("WMA", 6),
    # ("WMA", 7),
    # ("WMA", 8),
    # ("WMA", 9),
    # ("WMA", 10),
    # ("DCM", 4),
    # ("DCM", 5),
    # ("DCM", 6),
    # ("DCM", 7),
    # ("DCM", 8),
    # ("DCM", 9),
    # ("DCM", 10),
]

ENVELOPES = [[0.03, 0.05, 0.07]]
TIMEFRAMES = ["15m"]
STOP_LOSS = [0.3, 0.4]
PRICE_JUMP = [0.2, 0.3]
LEVERAGES = [2]
MODES = ['both', 'long', 'short']

# to avoid globals
def run_backtests():
    setups = []
    for coin in [
        # DOGE,
        FET,
        # TAO,
        # XRP,
        # INJ,
        # ONDO,
    ]:
        for mode in MODES:
            for leverage in LEVERAGES:
                for priceJump in PRICE_JUMP:
                    for stopLoss in STOP_LOSS:
                        for envelope in ENVELOPES:
                            for timeframe in TIMEFRAMES:
                                for avgType, avgPeriod in AVERAGES:
                                    setup = copy.deepcopy(coin)
                                    #            setup["start_date"] = "2024-04-12 00:00:00"
                                    setup["timeframe"] = timeframe
                                    setup["leverage"] = leverage
                                    setup["strategy_params"]["envelopes"] = envelope
                                    setup["strategy_params"]["stop_loss_pct"] = stopLoss
                                    setup["strategy_params"]["price_jump_pct"] = priceJump
                                    setup["strategy_params"]["average_type"] = avgType
                                    setup["strategy_params"]["average_period"] = avgPeriod
                                    setup["strategy_params"]["mode"] = mode
                                    setups.append(setup)
        #                        break
        #                    break
        #                 break
        #             break
        #         break
        #     break
        # break

    backtests = []
    for setup in setups:
        print(setup)
        backtest = envelope_backtest(setup)
        backtests.append(backtest)

    BacktestsToCSVFile(backtests, "../fet_backtests.csv")


run_backtests()

{'ticker': 'FET/USDT:USDT', 'timeframe': '15m', 'start_date': '2023-03-03 00:00:00', 'leverage': 2, 'strategy_params': {'average_type': 'EMA', 'average_period': 10, 'envelopes': [0.03, 0.05, 0.07], 'stop_loss_pct': 0.3, 'price_jump_pct': 0.2, 'position_size_percentage': 100, 'mode': 'both'}}
{'ticker': 'FET/USDT:USDT', 'timeframe': '15m', 'start_date': '2023-03-03 00:00:00', 'leverage': 2, 'strategy_params': {'average_type': 'EMA', 'average_period': 10, 'envelopes': [0.03, 0.05, 0.07], 'stop_loss_pct': 0.4, 'price_jump_pct': 0.2, 'position_size_percentage': 100, 'mode': 'both'}}
{'ticker': 'FET/USDT:USDT', 'timeframe': '15m', 'start_date': '2023-03-03 00:00:00', 'leverage': 2, 'strategy_params': {'average_type': 'EMA', 'average_period': 10, 'envelopes': [0.03, 0.05, 0.07], 'stop_loss_pct': 0.3, 'price_jump_pct': 0.3, 'position_size_percentage': 100, 'mode': 'both'}}
{'ticker': 'FET/USDT:USDT', 'timeframe': '15m', 'start_date': '2023-03-03 00:00:00', 'leverage': 2, 'strategy_params': {'